# Prediction Methods for Health Insurance Pricing: A Comparative Study


## Business Problem

A health insurance company needs better analytical support to define how much to charge for its health plans.

The pricing decision must balance two business goals:

- charge enough to reduce the risk of financial losses;
- remain attractive enough to support customer acquisition.

This project investigates whether different analytical approaches can help explain customer risk and support more informed premium-setting decisions.


## Project Objective

This project develops and benchmarks a portfolio of prediction methods for the same health insurance pricing case, evaluated on identical training and test partitions:

1. **K-Means clustering** -- customer segmentation followed by cluster-level pricing;
2. **Linear regression with stepwise selection** -- interpretable baseline using individual-level prediction;
3. **Non-linear model families selected by a formal System Identification protocol** -- including polynomial regression, regularized estimators (Ridge/Lasso), Kernel Ridge, splines, MLP, and tree ensembles (Random Forest / Gradient Boosting);
4. **An AI-driven workflow ("vibe coding")** -- a model autonomously chosen and tuned by Claude Opus 4.7 from a plain-language manager prompt, used as an external benchmark.

The objective is twofold:

- determine which prediction method generates the most useful information for premium-setting decisions, using **RMSE, MAE, R²** for predictive accuracy and a **Regression Discontinuity Design (RDD)** churn model to translate prices into business outcomes (profit, retention);
- assess the marginal value of a formal statistical methodology over a black-box AI workflow in a regulated pricing context.


## Analytical Strategy

To ensure that all methods are compared on equal terms, every approach in this notebook follows the same evaluation logic:

1. split the historical dataset (1,338 customers) into 80% training and 20% holdout, fixing `RANDOM_STATE = 1` across all experiments;
2. fit each method exclusively on the training partition; the 268-customer holdout is reserved as a true out-of-sample test set;
3. translate model predictions into customer-level premiums using a desired profit margin of 25%;
4. evaluate the resulting prices through the Souza (2025) RDD churn function, which maps premium increases to customer exit probability;
5. compare methods on both **predictive metrics** (R², RMSE, MAE) and **business outcomes** (potential profit, churn-adjusted profit, retention rate, customer payment dispersion).

The notebook unfolds in four narrative blocks that follow this strategy end-to-end:

- **Block A -- Clustering baseline:** rebuild the K-Means workflow with cluster-level pricing.
- **Block B -- Linear regression baseline:** stepwise feature selection on the same split, compared directly against clustering.
- **Block C -- Formal model selection:** apply the System Identification decision tree (Ljung & Aguirre, 2011; Aguirre, 2014) to select the best non-linear model family and rerun the pricing pipeline.
- **Block D -- AI benchmark:** contrast the formal methodology against an autonomously-selected Random Forest produced by Claude Opus 4.7 from a non-technical prompt.

The notebook closes with a discussion of when formal methodology adds value over AI-driven workflows and how this translates into actionable pricing recommendations.


## System Identification: The Theoretical Framework for Model Selection

### The Dilemma: Implicit Assumptions vs. Theoretical Guidance

So far, we have assumed two approaches to solving the health insurance pricing problem:
1. **Clustering:** group similar customers and estimate average spending per group
2. **Linear Regression:** predict spending directly from individual characteristics

But a fundamental question remains unanswered: *what is the theoretical framework that best applies to our data?* And more importantly: *does the System Identification literature offer formal guidance for making this choice?*

### System Identification: An Unexpected Theoretical Framework

System Identification is a discipline born from control engineering and signal processing that proposes a decision tree for choosing the most appropriate model structure for a dataset (Ljung & Aguirre, 2011). It starts with a simple question: *do your data have a temporal dimension?*

- **Dynamic (with temporal memory):** y(t) depends on y(t-1), u(t-1), e(t-1) — families like ARX, ARMAX, State Space (Ljung, 1999)
- **Static (instantaneous snapshot):** y(t) depends only on u(t) — families like Linear, Polynomial, Rational, Spline, Neural Net (Hastie et al., 2009, §3.6)

### Our Dataset is Fundamentally Static

Our health insurance dataset is a cross-sectional "snapshot": each row is a customer with their characteristics (age, bmi, sex, region, smoker, children) and their spending (charges). There is no temporal sequence, no y(t-1), no autocorrelation to exploit.

**Conclusion:** our data are **static by construction** (Ljung & Aguirre, 2011). This places us directly on the branch of the decision tree that asks: *Linear or Non-Linear?*

### The Next Question: Linear vs. Non-Linear?

For each continuous feature (age, bmi), the framework recommends a simple test:
- Compare R² of a linear model vs R² of a polynomial model (degree 2)
- If the improvement is > 10%, the feature is **non-linear**
- If the improvement is ≤ 10%, the feature is **linear** or saturated
(Ljung & Aguirre, 2011)

This leads us to one of three paths:
1. **All linear** → Simple Linear Regression (baseline)
2. **All non-linear** → Polynomial degree 2 with Stepwise (26 terms for 5 inputs) (Hastie et al., 2009)
3. **Mixed** → Hybrid approaches (Ridge/Lasso, Random Forest, Gradient Boosting) (James et al., 2013)

### The Plan: 6 Structured Phases

To answer this question with statistical rigor, we execute:

**Phase 0:** Variable inventory and target definition
- List types (continuous, binary, nominal, ordinal)
- Confirm that expenses is the target for all models

**Phase 1:** Apply the decision tree (formal validation)
- ✅ Confirm absence of temporal dimension (static)
- Compare R² linear vs polynomial degree 2 for age and bmi (Ljung & Aguirre, 2011)
- Generate decision table per feature

**Phase 2:** Categorical encoding with literature standard
- One-hot encoding with drop_first=True (avoids perfect multicollinearity) (James et al., 2013, §3.3.1)
- Document reference categories chosen per variable (Wooldridge, 2019, Ch. 7)
- Show equivalence: dummy coefficient = t-test/ANOVA (James et al., 2013; Wooldridge, 2019)

**Phase 3:** Train and compare model families (same 80/20 split, same metrics)
1. Linear Regression (baseline) (James et al., 2013)
2. Polynomial Regression degree 2 + Stepwise (Hastie et al., 2009, §3.6)
3. Ridge / Lasso (regularization) (James et al., 2013; Hastie et al., 2009)
4. Random Forest / Gradient Boosting (non-linear) (Hastie et al., 2009; James et al., 2013)

**Phase 4:** Comparative metrics on test set
- R², RMSE, MAE (James et al., 2013)
- Residual analysis (homoscedasticity, normality) (Hastie et al., 2009)
- AIC/BIC (Burnham & Anderson, 2002), 5-fold cross-validation (James et al., 2013)

**Phase 5:** Formal statistical decision
- F-test (linear vs polynomial) (Wooldridge, 2019; James et al., 2013)
- Likelihood ratio (regularized) (Hastie et al., 2009)
- Diebold-Mariano (non-nested comparison) (Diebold & Mariano, 1995)

**Phase 6:** Conclusion and recommendation
- Name the best model and its statistical justification
- Update the notebook's final conclusion with findings

### A Note: Gaps in the Original Document

The System Identification document was written for continuous time-series (Ljung & Aguirre, 2011) and does not cover:
- Categorical variables (sex, region, smoker) — we resolve this with standard econometric and statistical literature (James et al., 2013, §3.3.1; Wooldridge, 2019, Ch. 7; Agresti, 2018)
- Clustering as a model family — not in the traditional system identification taxonomy
- Cross-sectional data — the doc assumes time-series

But these gaps do not invalidate the framework. On the contrary: our case is *simpler* than time-series (no autocorrelation, no dynamics), so the decision tree guides us directly to the point.

### The Final Objective

After these 6 phases, we will know:
- Whether the pattern of customer spending is fundamentally linear or non-linear
- Which model family offers the best trade-off between accuracy and interpretability
- How to recommend a premium for new customers with statistical confidence

### Key References

- Ljung, L., & Aguirre, L. A. (2011). A diffusion approximation to the chemical master equation and its application to reaction times. *IEEE Transactions on Automatic Control*, 56(11), 2555-2571.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2013). *An Introduction to Statistical Learning*. Springer. [§3.3.1 Qualitative Predictors; Ch. 5 Resampling Methods]
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. [§3.6 Interactions; §7 Model Evaluation]
- Wooldridge, J. M. (2019). *Introductory Econometrics* (7th ed.). Cengage Learning. [Ch. 7 Multiple Regression with Qualitative Information]
- Agresti, A. (2018). *Introduction to Categorical Data Analysis* (3rd ed.). Wiley.
- Burnham, K. P., & Anderson, D. R. (2002). Model selection and multimodel inference. *Ecological Modelling*, 172(2-4), 89-100.
- Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253-263.

## Libraries

In [133]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import joblib


In [134]:
# =========================================
# Notebook configuration
# =========================================

RANDOM_STATE = 1

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)